# This notebook is used to build an Artificial Neural Network (MultiLayerPerceptron) model using the Diabetes Readmission dataset

## These are some of the prerequisites to run this notebook when running in the community edition of databricks

1. Create a New Cluster using the specifications from the quickstart notebook/guide
1. Import the diabetes_readmit_logreg_class1_csv dataset file from your system 
1. Attach the cluster you created in Step 1 to this notebook
1. When the cluster changes from <img src="http://docs.databricks.com/_static/images/clusters/cluster-starting.png"/></a> to <img src="http://docs.databricks.com/_static/images/clusters/cluster-running.png"/></a>, you are ready to run this notebook

In [0]:
# Data processing
from pyspark.sql.functions import log, col, exp

# Modeling
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import MultilayerPerceptronClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator


In [0]:
write_path = 'dbfs:/tmp/reproducible_ml_uofl/diab_readmit_uofl.delta'
diabetes_readmit = spark.read.format('delta').load(write_path)

#Show basic summary stats
display(diabetes_readmit.summary())

summary,patient_nbr,time_in_hospital,num_procedures,num_lab_procedures,num_medications,number_outpatient,number_inpatient,number_emergency,number_diagnoses,gender_cd,DiabetesMedication,readmit_flag,race_cd
count,101766,101766,101766,101766,101766,101766,101766,101766,101766,101766,101766,101766,101766
mean,5.4330400694947235E7,4.395986871843248,1.339730361810428,43.09564098028811,16.021844230882614,0.36935715268360747,0.635565906098304,0.19783621248747127,7.422606764538254,0.5375862272271682,0.7700312481575379,0.11159915885462728,null
stddev,3.869635934653421E7,2.985107767471267,1.705806979121172,19.674362249142096,8.127566209167309,1.2672650965326817,1.26286329009732,0.9304722684224632,1.9336001449974247,0.49858772375671534,0.420814525814695,0.3148741984505526,null
min,135,1,0,1,1,0,0,0,1,0,0,0,AfrAmr
25%,23412645,2,0,31,10,0,0,0,6,0,1,0,null
50%,45500490,4,1,44,15,0,0,0,8,1,1,0,null
75%,87532902,6,2,57,20,0,1,0,9,1,1,0,null
max,189502619,14,6,132,81,42,21,76,16,1,1,1,White


In [0]:
# Train test split
trainDF, testDF = diabetes_readmit.randomSplit([.65, .35], seed=42)
# Print the number of records
print(f'There are {trainDF.cache().count()} records in the training dataset.')
print(f'There are {testDF.cache().count()} records in the testing dataset.')

There are 66284 records in the training dataset.
There are 35482 records in the testing dataset.


##Now we need to modify the categorical variable race_cd into one-hot-encoded version
 For this we can either use the StringIndexer and OneHotEncoder separately OR use a pipeline to do this in one step


In [0]:
#You can also create a pipeline and do everything together in one easy fit and transform step
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, VectorAssembler
 
categoricalColumns = ["race_cd"]
stages = [] # stages in Pipeline
for categoricalCol in categoricalColumns:
    # Category Indexing with StringIndexer
    stringIndexer = StringIndexer(inputCol=categoricalCol, outputCol=categoricalCol + "Index")
    
# Use OneHotEncoder to convert categorical variables into binary SparseVectors
from pyspark.ml.feature import OneHotEncoder
encoder = OneHotEncoder(inputCols=[stringIndexer.getOutputCol()], outputCols=[categoricalCol + "classVec"])

# Define the pipeline based on the stages created in previous steps.
pipeline = Pipeline(stages=[stringIndexer, encoder])
 
# Define the pipeline model.
transform_mdl = pipeline.fit(trainDF)
trainDF21=transform_mdl.transform(trainDF)
trainDF21.show()

+-----------+----------------+--------------+------------------+---------------+-----------------+----------------+----------------+----------------+---------+------------------+------------+-------+------------+---------------+
|patient_nbr|time_in_hospital|num_procedures|num_lab_procedures|num_medications|number_outpatient|number_inpatient|number_emergency|number_diagnoses|gender_cd|DiabetesMedication|readmit_flag|race_cd|race_cdIndex|race_cdclassVec|
+-----------+----------------+--------------+------------------+---------------+-----------------+----------------+----------------+----------------+---------+------------------+------------+-------+------------+---------------+
|        135|               3|             1|                31|             14|                0|               1|               0|               5|        1|                 1|           0|  White|         0.0|  (4,[0],[1.0])|
|        135|               8|             6|                77|             33|    

In [0]:
# Linear regression expect a vector input
vecAssembler = VectorAssembler(inputCols=['time_in_hospital','num_procedures','num_medications', 'number_inpatient','number_emergency','number_diagnoses','DiabetesMedication','race_cdclassVec'], outputCol="features")
vecTrainDF = vecAssembler.transform(trainDF21)

In [0]:
# input layer of size 11 (features), one hidden layer of size 15, and an output layer of size 2 (classes)
layers = [11,15,2]

# Create Decision tree calssifier
ann_mlpc = (MultilayerPerceptronClassifier(labelCol='readmit_flag',
                                            featuresCol='features',
                                            maxIter=100,
                                            layers=layers,
                                            blockSize=128,
                                            seed=1234))
ann_mdl = ann_mlpc.fit(vecTrainDF)
predict_train = ann_mdl.transform(vecTrainDF)
predict_train.select('readmit_flag', 'rawPrediction', 'prediction', 'probability').show(10)


+------------+--------------------+----------+--------------------+
|readmit_flag|       rawPrediction|prediction|         probability|
+------------+--------------------+----------+--------------------+
|           0|[0.35908449431419...|       0.0|[0.87416633737338...|
|           1|[0.51208031920807...|       0.0|[0.88886547068572...|
|           0|[0.47149691320952...|       0.0|[0.89866187253150...|
|           0|[0.77052720808228...|       0.0|[0.93546099694184...|
|           0|[0.17696960887173...|       0.0|[0.82663726065148...|
|           0|[0.14931762895351...|       0.0|[0.84172312471059...|
|           1|[0.69500173900078...|       0.0|[0.90720305639722...|
|           0|[0.27595901206486...|       0.0|[0.89237508050660...|
|           0|[0.44025432094527...|       0.0|[0.88065722389722...|
|           0|[0.82401398671989...|       0.0|[0.93910817758543...|
+------------+--------------------+----------+--------------------+
only showing top 10 rows



In [0]:
testDF21=transform_mdl.transform(testDF) #do the data transformation using saved parameters from training
vecTestDF = vecAssembler.transform(testDF21) #do the feature transformation using vector assembler
# Make predictions on testing dataset
predict_test = ann_mdl.transform(vecTestDF) #make predictions using the trained model

In [0]:
eval_ann = BinaryClassificationEvaluator(labelCol = "readmit_flag")
auc_train = eval_ann.evaluate(predict_train)
print(auc_train)

auc_test = eval_ann.evaluate(predict_test)
print(auc_test)

0.6237841985381569
0.6165414693771745


In [0]:
import pandas as pd
import time 
import sys
from pyspark.sql.window import *
from pyspark.sql.types import * 
from pyspark.sql.functions import *
import pyspark.sql.functions as f

def get_model_stats(print_lable,dataset,evaluator,mbr_id,target):
  print(print_lable)
  AUROC=evaluator.evaluate(dataset)
  
  tot_event=dataset.count()
  tot_target=dataset.select(target).filter(f.col(target)==1).count()
  inc_rate=float(tot_target)/float(tot_event)
  print('Incidence rate in dataset: ',inc_rate)
  print('AUROC: ',AUROC)
  

  def mbr_prob1(prob_vec):
    return float(prob_vec[1]) #Get the second value from probability vector of the prediction since that is probability of target 1
  def mbr_prob2(rawp_vec):
    prob_vec = np.exp(rawp_vector)/(1+ np.exp(rawp_vec)) #this is the same as the logit function e**x/1+e**x
    return float(prob_vec[1])
  get_prob_1= udf(mbr_prob1, FloatType())
  get_prob_2= udf(mbr_prob2, FloatType())

  try:
    df_plot_tmp = dataset.withColumn('prob_score', get_prob_1('probability'))
  except:
    df_plot_tmp = dataset.withColumn('prob_score', get_prob_2('rawPrediction'))
  df_reg1=(df_plot_tmp.select(mbr_id,"rawPrediction","probability","prob_score",target))

  #Create Percentiles based on the predicted probability
  df_reg_12=  df_reg1.sort(col("prob_score").desc())
  df_reg_12 = df_reg_12.withColumn("new_column",lit("ABC"))
  w = Window().partitionBy("new_column").orderBy(col("prob_score").desc())  
  df_reg_12 = df_reg_12.withColumn("row_num",row_number().over(w)).drop("new_column")
  df_reg2 = (df_reg_12
             .withColumn("score_pctl", f.percent_rank().over(Window.orderBy(f.col("row_num"))))
             .withColumn("pctl_cat", f.when(f.col("score_pctl")==1, f.lit(100)).otherwise(f.floor(f.col("score_pctl")*100) + 1))
             )
  df_reg22 = (df_reg2
              .groupBy("pctl_cat")
              .agg(f.mean(target).alias("pctl_target_rate"),\
                   (f.sum(target)).alias("pctl_target_total"),\
                   (f.count(mbr_id)).alias("total_count"),\
                   (f.mean("prob_score")).alias("mean_predicted_prob")
                  ,)
              .orderBy(f.col("pctl_cat"))
              )

  #Now let's create the cumulative percentiles
  cum_sum = (df_reg22
              .withColumn('cum_target', f.sum(df_reg22.pctl_target_total).over(Window.partitionBy().orderBy().rowsBetween(-sys.maxsize, 0)))
              .withColumn('cum_total', f.sum(df_reg22.total_count).over(Window.partitionBy().orderBy().rowsBetween(-sys.maxsize, 0)))
              .withColumn('cum_target_cap_rate',f.col("cum_target")/f.col("cum_total"))
              .withColumn('cum_target_cap_of_total',((f.col("cum_target")/tot_target))*100)
            )
  return cum_sum

In [0]:
cum_cap_dt = get_model_stats("Artificial Neural Network",predict_test,eval_ann,"patient_nbr","readmit_flag")

Artificial Neural Network
Incidence rate in dataset:  0.11225410067076264
AUROC:  0.6165414693771745


In [0]:
display(cum_cap_dt)

pctl_cat,pctl_target_rate,pctl_target_total,total_count,mean_predicted_prob,cum_target,cum_total,cum_target_cap_rate,cum_target_cap_of_total
1,0.4140845070422535,147,355,0.3826438638525949,147,355,0.4140845070422535,3.690685413005272
2,0.3352112676056338,119,355,0.3127895175571173,266,710,0.37464788732394366,6.6783831282952555
3,0.24225352112676057,86,355,0.25849904271078783,352,1065,0.3305164319248826,8.837559628420788
4,0.22253521126760564,79,355,0.2312270507006578,431,1420,0.30352112676056336,10.820989204117499
5,0.21971830985915494,78,355,0.21679030232866045,509,1775,0.28676056338028166,12.779312076324379
6,0.20903954802259886,74,354,0.20864387037558743,583,2129,0.2738374823860968,14.63720813457193
7,0.18873239436619718,67,355,0.20110507422769575,650,2484,0.2616747181964573,16.31935726839066
8,0.18309859154929578,65,355,0.19203726032250365,715,2839,0.25184924269108844,17.951292995229725
9,0.22535211267605634,80,355,0.1859025544683698,795,3194,0.24890419536631184,19.95982927441627
10,0.18309859154929578,65,355,0.18157203902660962,860,3549,0.24232178078331926,21.591765001255336



## Assignment 3.1

1. Create a NNET with Two hidden layers with 10 Neurons in first hidden layer and 7 Neurons in second hidden layer
1. What is the test AUC for the new model?